# Capital Bikeshare (DC)

Fetches trip data straight from the public S3 bucket via `bike_network_traffic.get_bike_data` and links a Celldega clustergram to the deck.gl flow map.

In [ ]:
from bike_network_traffic import silence_warnings
silence_warnings()

import celldega as dega
from ipywidgets import HBox, Layout

from bike_network_traffic import (
    get_bike_data,
    link_flow_to_clustergram,
    make_flow_widget,
    make_station_clustergram,
)

## 1. Download trips and build station + transition tables

First call downloads (and caches under `~/.cache/bike_network_traffic`) the monthly zip from S3 and returns a station table plus a destination-probability matrix ready for Celldega.

In [ ]:
ds = get_bike_data('dc', year=2026, month=3, return_trips=True)
stations, transition_prob, trips = ds.stations, ds.transition_prob, ds.trips
print('stations:', stations.shape, ' transition_prob:', transition_prob.shape, ' trips:', trips.shape)
stations.head()

## 2. Cluster the transition matrix

In [ ]:
mat, cgm, cluster_map = make_station_clustergram(transition_prob, n_clusters=100)

## 3. Build the flow map and link it to the clustergram

In [ ]:
flow = make_flow_widget(stations, transition_prob, cluster_map, trips=trips, height=700, debug=True)
link_flow_to_clustergram(flow, cgm)

flow.layout = Layout(width='560px', height='700px')
cgm.layout  = Layout(width='720px', height='700px')
HBox([flow, cgm])